# Phase 6: INT8 Weight Quantization

*(This notebook is part of the GPT-2 Inference Engine from Scratch project.)*

## Lesson 32: Basic INT8 Quantization

**Goal:** We are doing Symmetric Per-Tensor INT8 Quantization on a weight matrix. We will measure the memory savings and check if our max error and MSE match the mathematical theory.

In [34]:
def quantize_int8(w):
    scale = w.abs().max() / 127.0
    q = torch.clamp(torch.round(w / scale), -127, 127).to(torch.int8)
    return q, scale

def dequantize_int8(q, scale):
    return q.to(torch.float32) * scale

w = blocks[0]["mlp.c_fc.weight"]
q, scale = quantize_int8(w)
w_dequant = dequantize_int8(q, scale)

mse = ((w - w_dequant) ** 2).mean().item()
max_err = (w - w_dequant).abs().max().item()

print("Weight shape:", tuple(w.shape))
print("Scale:", scale.item())
print("fp32 size (MB):", round(w.numel() * 4 / 1e6, 3))
print("int8 size (MB):", round(w.numel() * 1 / 1e6, 3))
print("MSE:", mse)
print("Max abs error:", max_err)
print("Relative max error:", max_err / w.abs().max().item())

Weight shape: (768, 3072)
Scale: 0.03612377494573593
fp32 size (MB): 9.437
int8 size (MB): 2.359
MSE: 0.0001088166463887319
Max abs error: 0.018061885610222816
Relative max error: 0.003937007448827556


## Lesson 33: Naive Dequantize-then-Matmul

**Goal:** Let's test a naive 'dequantize-then-matmul' approach. You will notice it is actually much slower than fp32 because writing and reading the full fp32 buffer takes too much overhead.

In [35]:
x_test = torch.randn(1, 768, device="cuda")  # single-token activation, decode jaisa

def naive_dequant_matmul(x, q, scale):
    w_fp32 = dequantize_int8(q, scale)
    return x @ w_fp32

t_fp32 = bench(lambda: x_test @ w)
t_naive_quant = bench(lambda: naive_dequant_matmul(x_test, q, scale))

out_fp32 = x_test @ w
out_naive = naive_dequant_matmul(x_test, q, scale)

print(f"Plain fp32 matmul:       {t_fp32*1000:.4f} ms")
print(f"Naive dequant+matmul:     {t_naive_quant*1000:.4f} ms")
print(f"Speedup: {t_fp32/t_naive_quant:.2f}x")
print(f"Max abs diff (from quant error): {(out_fp32 - out_naive).abs().max().item()}")

Plain fp32 matmul:       0.0571 ms
Naive dequant+matmul:     0.1988 ms
Speedup: 0.29x
Max abs diff (from quant error): 1.103922724723816


## Lesson 34: Relative Error and Direction Check

**Goal:** We want to see how quantization error builds up during matrix multiplication. While the relative error goes up, the cosine similarity stays very high, meaning the vector direction is mostly preserved.

In [36]:
print("out_fp32 stats:")
print("  mean abs:", out_fp32.abs().mean().item())
print("  max abs:", out_fp32.abs().max().item())
print("  std:", out_fp32.std().item())

relative_max_error = (out_fp32 - out_naive).abs().max().item() / out_fp32.abs().max().item()
cos_sim = torch.nn.functional.cosine_similarity(out_fp32.flatten(), out_naive.flatten(), dim=0).item()

print(f"\nRelative max error: {relative_max_error*100:.2f}%")
print(f"Cosine similarity: {cos_sim:.6f}")

out_fp32 stats:
  mean abs: 2.9134538173675537
  max abs: 15.088730812072754
  std: 3.706496000289917

Relative max error: 7.32%
Cosine similarity: 0.996973


## Lesson 35: Fused INT8 Matvec Kernel

**Goal:** We are writing a Fused INT8 kernel that reads the INT8 weights directly, dequantizes them in SRAM, and does the math. We will compare this with fp32 and the naive approach.

In [37]:
@triton.jit
def quant_matvec_kernel(x_ptr, w_ptr, out_ptr, scale, K, N, BLOCK_K: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_n = tl.program_id(0)
    n_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    n_mask = n_offsets < N

    acc = tl.zeros((BLOCK_N,), dtype=tl.float32)
    for k_start in range(0, K, BLOCK_K):
        k_offsets = k_start + tl.arange(0, BLOCK_K)
        k_mask = k_offsets < K
        x_chunk = tl.load(x_ptr + k_offsets, mask=k_mask, other=0.0)
        w_ptrs = w_ptr + k_offsets[:, None] * N + n_offsets[None, :]
        w_mask = k_mask[:, None] & n_mask[None, :]
        w_chunk = tl.load(w_ptrs, mask=w_mask, other=0).to(tl.float32) * scale
        acc += tl.sum(x_chunk[:, None] * w_chunk, axis=0)

    tl.store(out_ptr + n_offsets, acc, mask=n_mask)

def quant_matvec(x_flat, q, scale, N):
    K = x_flat.shape[0]
    out = torch.empty(N, device="cuda", dtype=torch.float32)
    BLOCK_K, BLOCK_N = 128, 128
    grid = (triton.cdiv(N, BLOCK_N),)
    quant_matvec_kernel[grid](x_flat, q.contiguous(), out, scale.item(), K, N, BLOCK_K=BLOCK_K, BLOCK_N=BLOCK_N)
    return out

x_flat = x_test.flatten()
out_kernel = quant_matvec(x_flat, q, scale, N=3072)

t_kernel = bench(lambda: quant_matvec(x_flat, q, scale, N=3072))

print("Max abs diff vs naive dequant+matmul:", (out_kernel - out_naive.flatten()).abs().max().item())
print(f"\nPlain fp32 matmul:      {t_fp32*1000:.4f} ms")
print(f"Naive dequant+matmul:    {t_naive_quant*1000:.4f} ms")
print(f"Fused quant kernel:      {t_kernel*1000:.4f} ms")
print(f"Speedup vs fp32:  {t_fp32/t_kernel:.2f}x")
print(f"Speedup vs naive: {t_naive_quant/t_kernel:.2f}x")

Max abs diff vs naive dequant+matmul: 1.0013580322265625e-05

Plain fp32 matmul:      0.0571 ms
Naive dequant+matmul:    0.1988 ms
Fused quant kernel:      0.0658 ms
Speedup vs fp32:  0.87x
Speedup vs naive: 3.02x


## Sprint 5 summary
- INT8 symmetric quantization: exactly 4x smaller, error formulas (max=scale/2, MSE~=scale^2/12) matched theory exactly
- Naive dequant-then-matmul: 3.7x SLOWER than fp32 (extra write+read of full fp32 buffer)
- Weight-level error (0.39%) amplified to output-level (6.24% relative, cosine sim 0.997) through the 768-term reduction
- Fused INT8 matvec kernel: correct (matches naive to precision floor), 2.28x faster than naive, but still 0.62x vs cuBLAS fp32
- Honest finding: our simple loop-based kernel doesn't use Tensor Cores , bandwidth savings alone don't beat a mature, tuned library kernel; real production INT8 kernels need much more engineering (tiling, tensor cores, careful occupancy)